In [3]:
import math

import torch
import plotly.express as px
from torch import nn, Tensor
from torch.optim import Optimizer, AdamW
from torch.optim.lr_scheduler import LRScheduler, LambdaLR
from torch.utils.data import TensorDataset, DataLoader

from mae.model import *
from src import dataset
from src import metrics
from src import configs as cfg
from mae.utils import setup_seed

/root/repos/raidium_challenge/.venv/lib/python3.13/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [6]:
def main():
    max_lr=1e-3
    mask_ratio=0.75

    train_cfg = cfg.TrainingConfig(
        n_epochs=5000,
        batch_size=64,
    )
    train_cfg.warmup_epochs = 500
    model_cfg = cfg.ModelConfig()
    model_cfg.mask_ratio = 0.75
    dataset_cfg = cfg.DatasetConfig()

    dataset.mk_dataset(verbose=False)
    setup_seed(train_cfg.random_state)
    torch.backends.cuda.matmul.fp32_precision = 'ieee'

    x_train, y_train, x_test = dataset.load_raw_dataset(cfg.DEVICE)
    train_tensor = torch.cat((x_train, x_test)).to(device="cuda")
    train_ds = torch.utils.data.TensorDataset(train_tensor)
    train_loader = torch.utils.data.DataLoader(
        train_ds,
        train_cfg.batch_size,
        shuffle=True,
    )

    model = MAE_ViT(
        image_size=256,
        mask_ratio=mask_ratio,
        patch_size=16,
        emb_dim=256,
        encoder_layer=8,
        encoder_head=8,
        decoder_head=8,
    ).to(cfg.DEVICE)
    model.cfg = model_cfg
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=max_lr * train_cfg.batch_size / 256,
        betas=(0.9, 0.95),
    )
    def lr_func(epoch: int) -> float:
        return min(
            (epoch + 1) / (train_cfg.warmup_epochs + 1e-8),
            0.5 * (math.cos(epoch / train_cfg.n_epochs * math.pi) + 1)
        )
    lr_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_func)
    train_model(model, train_cfg, optimizer, lr_scheduler, train_loader, train_loader)

def train_model(
        model: torch.nn.Module,
        train_cfg: cfg.TrainingConfig,
        optimizer: torch.optim.Optimizer,
        lr_scheduler: torch.optim.lr_scheduler.LRScheduler,
        train_loader: DataLoader,
        val_loader: DataLoader,
    ) -> dict[str, Tensor]:
    # eval_model(model, val_loader)
    for e in range(train_cfg.n_epochs):
        epoch_dict = train_model_for_single_epoch(model, optimizer, lr_scheduler, train_loader)
        print(f"epoch {e:3d}:", epoch_dict["loss"])
        if e % 50 == 0 or e == train_cfg.n_epochs - 1:
            eval_model(model, val_loader)

def train_model_for_single_epoch(
        model: torch.nn.Module,
        optimizer: torch.optim.Optimizer,
        lr_scheduler: torch.optim.lr_scheduler.LRScheduler,
        train_loader: torch.utils.data.DataLoader,
    ) -> dict[str, Tensor]:
    model.train()
    losses = []
    for (x,) in train_loader:
        x = dataset.preprocess_imgs(x)
        step_dict = perform_training_step(model, x, optimizer)
        losses.append(step_dict["loss"].item())
    lr_scheduler.step()
    avg_loss = sum(losses) / len(losses)
    return {"loss": avg_loss}

def perform_training_step(model: torch.nn.Module, x: Tensor, otpimizer: torch.optim.Optimizer) -> dict[str, Tensor]:
    otpimizer.zero_grad()
    with torch.autocast(cfg.DEVICE.type, torch.bfloat16):
        predicted_img, mask = model(x)
        print("predicted_img:", predicted_img.shape)
        print("mask:", mask.shape)
        loss = torch.mean((predicted_img - x) ** 2 * mask) / model.cfg.mask_ratio
    loss.backward()
    otpimizer.step()
    return {"loss": loss}

@torch.no_grad
def eval_model(
        model: torch.nn.Module,
        data_loader: torch.utils.data.DataLoader,
    ):
    model.eval()
    (x, ) = next(iter(data_loader))
    x = dataset.preprocess_imgs(x)
    N_IMGS_TO_PLT = 5
    x = x[:N_IMGS_TO_PLT]
    predicted_val_img, mask = model(x)
    predicted_val_img = predicted_val_img * mask + x * (1 - mask)
    img = torch.cat([x * (1 - mask), predicted_val_img, x], dim=0)
    img = img * dataset.STD + dataset.MEAN 
    print(img.detach().cpu().numpy().shape)
    np_imgs = (
        img
        .detach()
        .cpu()
        .numpy()
        .squeeze()
    )
    fig = px.imshow(
        np_imgs,
        facet_col=0,
        facet_col_wrap=N_IMGS_TO_PLT,
        color_continuous_scale="rainbow",
    )
    display(fig)
    # TODO: return loss on val loader.
main()

predicted_img: torch.Size([64, 1, 256, 256])
mask: torch.Size([64, 1, 256, 256])
predicted_img: torch.Size([64, 1, 256, 256])
mask: torch.Size([64, 1, 256, 256])
predicted_img: torch.Size([64, 1, 256, 256])
mask: torch.Size([64, 1, 256, 256])
predicted_img: torch.Size([64, 1, 256, 256])
mask: torch.Size([64, 1, 256, 256])
predicted_img: torch.Size([64, 1, 256, 256])
mask: torch.Size([64, 1, 256, 256])
predicted_img: torch.Size([64, 1, 256, 256])
mask: torch.Size([64, 1, 256, 256])
predicted_img: torch.Size([64, 1, 256, 256])
mask: torch.Size([64, 1, 256, 256])
predicted_img: torch.Size([64, 1, 256, 256])
mask: torch.Size([64, 1, 256, 256])
predicted_img: torch.Size([64, 1, 256, 256])
mask: torch.Size([64, 1, 256, 256])
predicted_img: torch.Size([64, 1, 256, 256])
mask: torch.Size([64, 1, 256, 256])
predicted_img: torch.Size([64, 1, 256, 256])
mask: torch.Size([64, 1, 256, 256])
predicted_img: torch.Size([64, 1, 256, 256])
mask: torch.Size([64, 1, 256, 256])
predicted_img: torch.Size([6

predicted_img: torch.Size([64, 1, 256, 256])
mask: torch.Size([64, 1, 256, 256])
predicted_img: torch.Size([64, 1, 256, 256])
mask: torch.Size([64, 1, 256, 256])
predicted_img: torch.Size([64, 1, 256, 256])
mask: torch.Size([64, 1, 256, 256])
predicted_img: torch.Size([64, 1, 256, 256])
mask: torch.Size([64, 1, 256, 256])
predicted_img: torch.Size([64, 1, 256, 256])
mask: torch.Size([64, 1, 256, 256])


KeyboardInterrupt: 